# Qwen3-4B MAP Training Notebook

This notebook is structured for Kaggle + H100.

- Start with `QUICK_RUN = True` to sanity-check the pipeline.
- Switch to `QUICK_RUN = False` for full training.
- The trained model is saved under the model's own name.


In [ ]:
!pip install -q -U "transformers>=4.55.0" peft accelerate bitsandbytes datasets scikit-learn joblib


In [ ]:
import gc
import json
import inspect
import os
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42

TRAIN_PATH = "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/train.csv"
TEST_PATH = "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/test.csv"
OUTPUT_ROOT = Path("/kaggle/working/map_multi_llm_runs")

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_SLUG = MODEL_NAME.split("/")[-1]
HF_TOKEN = os.getenv("HF_TOKEN")

QUICK_RUN = True
QUICK_RUN_ROWS = 4096
QUICK_RUN_TEST_SIZE = 0.10
QUICK_RUN_EPOCHS = 1
QUICK_RUN_MAX_STEPS = 40
FULL_TRAIN_EPOCHS = 3

USE_4BIT = False
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else (torch.float16 if torch.cuda.is_available() else torch.float32)
OPTIMIZER_NAME = "adamw_torch"

MAX_LENGTH = 256
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
PER_DEVICE_TRAIN_BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4

LORA_R = 512
LORA_ALPHA = 32
LORA_DROPOUT = 0.3
TARGET_MODULES = ["q_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
MODULES_TO_SAVE = ["score"]

RUN_NAME = "quick_run" if QUICK_RUN else "full_train"
RUN_DIR = OUTPUT_ROOT / MODEL_SLUG / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
LOG_DIR = RUN_DIR / "logs"
FINAL_MODEL_DIR = RUN_DIR / MODEL_SLUG

for path in [OUTPUT_ROOT, RUN_DIR, CHECKPOINT_DIR, LOG_DIR, FINAL_MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print({
    "train_path": TRAIN_PATH,
    "test_path": TEST_PATH,
    "model_name": MODEL_NAME,
    "quick_run": QUICK_RUN,
    "use_4bit": USE_4BIT,
    "model_dtype": str(MODEL_DTYPE).replace("torch.", ""),
    "gpu_count": torch.cuda.device_count(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "run_dir": str(RUN_DIR),
    "final_model_dir": str(FINAL_MODEL_DIR),
})


In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def format_input_v2(row):
    is_correct_text = "Yes" if row["is_correct"] == 1 else "No"
    return (
        f"Question: {row['QuestionText']}\n"
        f"Answer: {row['MC_Answer']}\n"
        f"Is Correct Answer: {is_correct_text}\n"
        f"Student Explanation: {row['StudentExplanation']}"
    )


def compute_map3(eval_pred):
    logits, labels = eval_pred
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()
    top3 = np.argsort(-probs, axis=1)[:, :3]
    matches = top3 == labels[:, None]

    score = 0.0
    for i in range(len(labels)):
        if matches[i, 0]:
            score += 1.0
        elif matches[i, 1]:
            score += 0.5
        elif matches[i, 2]:
            score += 1.0 / 3.0

    return {"map@3": score / len(labels)}


def build_training_frame(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Misconception"] = df["Misconception"].fillna("NA")
    df["target"] = df["Category"] + ":" + df["Misconception"]

    idx = df["Category"].str.split("_").str[0] == "True"
    correct = df.loc[idx].copy()
    correct["c"] = correct.groupby(["QuestionId", "MC_Answer"])["MC_Answer"].transform("count")
    correct = correct.sort_values("c", ascending=False)
    correct = correct.drop_duplicates(["QuestionId"])
    correct = correct[["QuestionId", "MC_Answer"]]
    correct["is_correct"] = 1

    df = df.merge(correct, on=["QuestionId", "MC_Answer"], how="left")
    df["is_correct"] = df["is_correct"].fillna(0).astype(np.int64)
    return df


def dataframe_to_dataset(frame: pd.DataFrame, tokenizer, max_length: int) -> Dataset:
    clean = frame[["text", "label"]].copy()
    clean["label"] = clean["label"].astype(np.int64)
    ds = Dataset.from_pandas(clean, preserve_index=False)
    ds = ds.rename_column("label", "labels")

    def tokenize_fn(batch):
        return tokenizer(batch["text"], truncation=True, max_length=max_length)

    return ds.map(tokenize_fn, batched=True, remove_columns=["text"])


seed_everything(SEED)


In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

train = build_training_frame(train)

le = LabelEncoder()
train["label"] = le.fit_transform(train["target"])
n_classes = len(le.classes_)
id2label = {i: label for i, label in enumerate(le.classes_)}
label2id = {label: i for i, label in id2label.items()}

train["text"] = train.apply(format_input_v2, axis=1)

work_df = train.copy()
if QUICK_RUN:
    sample_size = min(QUICK_RUN_ROWS, len(work_df))
    work_df = work_df.sample(n=sample_size, random_state=SEED).reset_index(drop=True)
    stratify_labels = work_df["label"] if work_df["label"].value_counts().min() >= 2 else None
    train_df, eval_df = train_test_split(
        work_df,
        test_size=QUICK_RUN_TEST_SIZE,
        random_state=SEED,
        stratify=stratify_labels,
    )
else:
    train_df, eval_df = work_df, None

train_df = train_df.reset_index(drop=True)
if eval_df is not None:
    eval_df = eval_df.reset_index(drop=True)

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Working train shape: {train_df.shape}")
print(f"Validation shape: {None if eval_df is None else eval_df.shape}")
print(f"Target classes: {n_classes}")
print("\nExample formatted prompt:\n")
print(train_df.loc[0, "text"])


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

train_ds = dataframe_to_dataset(train_df, tokenizer, MAX_LENGTH)
eval_ds = dataframe_to_dataset(eval_df, tokenizer, MAX_LENGTH) if eval_df is not None else None
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

print(train_ds)
print(eval_ds)


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("This notebook is intended for a GPU runtime. Kaggle H100 is recommended.")

torch.cuda.set_device(0)
print(f"Using GPU: {torch.cuda.get_device_name(0)}")
print(f"Visible CUDA devices: {torch.cuda.device_count()}")

model_load_kwargs = {
    "pretrained_model_name_or_path": MODEL_NAME,
    "num_labels": n_classes,
    "torch_dtype": MODEL_DTYPE,
    "token": HF_TOKEN,
    "trust_remote_code": True,
}

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model_load_kwargs["quantization_config"] = bnb_config
    model_load_kwargs["device_map"] = "auto"

model = AutoModelForSequenceClassification.from_pretrained(**model_load_kwargs)

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)
else:
    model = model.to("cuda")

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.config.id2label = id2label
model.config.label2id = label2id
model.config.problem_type = "single_label_classification"

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="SEQ_CLS",
    modules_to_save=MODULES_TO_SAVE,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
epochs = QUICK_RUN_EPOCHS if QUICK_RUN else FULL_TRAIN_EPOCHS
max_steps = QUICK_RUN_MAX_STEPS if QUICK_RUN else -1

training_arg_values = {
    "output_dir": str(CHECKPOINT_DIR),
    "overwrite_output_dir": True,
    "do_train": True,
    "do_eval": eval_ds is not None,
    "evaluation_strategy": "steps" if eval_ds is not None else "no",
    "eval_steps": 20 if eval_ds is not None else None,
    "save_strategy": "no",
    "logging_strategy": "steps",
    "logging_steps": 10 if QUICK_RUN else 100,
    "num_train_epochs": epochs,
    "max_steps": max_steps,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "lr_scheduler_type": "cosine",
    "remove_unused_columns": False,
    "bf16": USE_BF16,
    "fp16": torch.cuda.is_available() and not USE_BF16,
    "report_to": "none",
    "disable_tqdm": True,
    "gradient_checkpointing": True,
    "dataloader_drop_last": False,
    "dataloader_pin_memory": False,
    "optim": OPTIMIZER_NAME,
}

training_arg_signature = inspect.signature(TrainingArguments.__init__).parameters

if "eval_strategy" in training_arg_signature and "evaluation_strategy" not in training_arg_signature:
    training_arg_values["eval_strategy"] = training_arg_values.pop("evaluation_strategy")

supported_training_args = {
    key: value
    for key, value in training_arg_values.items()
    if key in training_arg_signature and value is not None
}

skipped_training_args = [
    key for key in training_arg_values if key not in supported_training_args and key not in training_arg_signature
]
if skipped_training_args:
    print(f"Skipping unsupported TrainingArguments keys: {sorted(skipped_training_args)}")

training_args = TrainingArguments(**supported_training_args)

trainer_signature = inspect.signature(Trainer.__init__).parameters

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_ds,
    "data_collator": data_collator,
}

if "processing_class" in trainer_signature:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature:
    trainer_kwargs["tokenizer"] = tokenizer

if eval_ds is not None:
    trainer_kwargs["eval_dataset"] = eval_ds
    trainer_kwargs["compute_metrics"] = compute_map3

supported_trainer_kwargs = {
    key: value
    for key, value in trainer_kwargs.items()
    if key in trainer_signature and value is not None
}

skipped_trainer_kwargs = [
    key for key in trainer_kwargs if key not in supported_trainer_kwargs and key not in trainer_signature
]
if skipped_trainer_kwargs:
    print(f"Skipping unsupported Trainer keys: {sorted(skipped_trainer_kwargs)}")

trainer = Trainer(**supported_trainer_kwargs)

for callback in list(trainer.callback_handler.callbacks):
    if callback.__class__.__name__ == "NotebookProgressCallback":
        trainer.remove_callback(callback.__class__)
        print("Removed NotebookProgressCallback for compatibility.")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_result = trainer.train()
metrics = dict(train_result.metrics)

if eval_ds is not None:
    metrics.update(trainer.evaluate())

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))
joblib.dump(le, RUN_DIR / "label_encoder.joblib")

adapter_inference_config = {
    "artifact_type": "peft_lora_adapter",
    "base_model_name": MODEL_NAME,
    "adapter_path": str(FINAL_MODEL_DIR),
    "label_encoder_path": str(RUN_DIR / "label_encoder.joblib"),
    "max_length": MAX_LENGTH,
    "use_4bit_for_training": USE_4BIT,
    "torch_dtype": str(MODEL_DTYPE).replace("torch.", ""),
}

with open(RUN_DIR / "adapter_inference_config.json", "w", encoding="utf-8") as f:
    json.dump(adapter_inference_config, f, indent=2)

summary = {
    "model_name": MODEL_NAME,
    "model_slug": MODEL_SLUG,
    "quick_run": QUICK_RUN,
    "saved_artifact_type": "peft_lora_adapter",
    "base_model_name": MODEL_NAME,
    "use_4bit": USE_4BIT,
    "torch_dtype": str(MODEL_DTYPE).replace("torch.", ""),
    "train_path": TRAIN_PATH,
    "test_path": TEST_PATH,
    "run_dir": str(RUN_DIR),
    "final_model_dir": str(FINAL_MODEL_DIR),
    "n_classes": n_classes,
    "train_rows": int(len(train_df)),
    "eval_rows": int(0 if eval_df is None else len(eval_df)),
    "metrics": metrics,
}

with open(RUN_DIR / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Training complete.")
print(f"Saved LoRA adapter to: {FINAL_MODEL_DIR}")
print(f"Base model required for inference: {MODEL_NAME}")
print(f"Saved label encoder to: {RUN_DIR / 'label_encoder.joblib'}")
print(json.dumps(metrics, indent=2))


In [ ]:
inference_adapter_path = FINAL_MODEL_DIR
inference_label_encoder_path = RUN_DIR / "label_encoder.joblib"
inference_device = "cuda" if torch.cuda.is_available() else "cpu"

inference_le = joblib.load(inference_label_encoder_path)
inference_tokenizer = AutoTokenizer.from_pretrained(
    inference_adapter_path,
    token=HF_TOKEN,
    trust_remote_code=True,
)

inference_model_load_kwargs = {
    "pretrained_model_name_or_path": MODEL_NAME,
    "num_labels": len(inference_le.classes_),
    "torch_dtype": MODEL_DTYPE,
    "token": HF_TOKEN,
    "trust_remote_code": True,
}

if USE_4BIT:
    inference_model_load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    inference_model_load_kwargs["device_map"] = "auto"

inference_base_model = AutoModelForSequenceClassification.from_pretrained(**inference_model_load_kwargs)
inference_base_model.config.pad_token_id = inference_tokenizer.pad_token_id
if not USE_4BIT and inference_device == "cuda":
    inference_base_model = inference_base_model.to(inference_device)

inference_model = PeftModel.from_pretrained(inference_base_model, inference_adapter_path)
inference_model.eval()

sample_text = train_df.loc[0, "text"]
inputs = inference_tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
if inference_device == "cuda":
    inputs = {key: value.to(inference_device) for key, value in inputs.items()}

with torch.no_grad():
    logits = inference_model(**inputs).logits

top_k = min(3, logits.shape[-1])
top3 = torch.topk(logits, k=top_k, dim=-1).indices[0].cpu().numpy()
print("Sample top predictions:")
print(inference_le.inverse_transform(top3))
